In [1]:
!pip install chromadb gradio sentence-transformers -q
print("Done!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 84.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 93.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

In [2]:
#Load CUAD
import json, re, random
import pandas as pd

cuad_path = "/kaggle/input/datasets/ashyou09/contract-understanding-atticus-dataset-cuad/CUAD_v1.json"
with open(cuad_path) as f:
    cuad_raw = json.load(f)

pairs = []
for contract in cuad_raw["data"]:
    for paragraph in contract["paragraphs"]:
        context = paragraph["context"]
        for qa in paragraph["qas"]:
            pairs.append({
                "query":   qa["question"],
                "passage": context,
                "label":   1 if qa["answers"] else 0
            })

df = pd.DataFrame(pairs)
print(f"Loaded: {len(df):,} pairs")

Loaded: 20,910 pairs


In [3]:
#Transform + split
query_mapping = {
    "Document Name":                     ("What is the name of this contract document?",              "Tên tài liệu hợp đồng này là gì?"),
    "Parties":                           ("Who are the parties involved in this contract?",            "Các bên tham gia hợp đồng này là ai?"),
    "Agreement Date":                    ("What is the agreement date of this contract?",              "Ngày ký kết hợp đồng là khi nào?"),
    "Effective Date":                    ("What is the effective date of this contract?",              "Ngày có hiệu lực của hợp đồng là khi nào?"),
    "Expiration Date":                   ("When does this contract expire?",                           "Hợp đồng này hết hạn vào ngày nào?"),
    "Renewal Term":                      ("What is the renewal term of this contract?",                "Điều khoản gia hạn hợp đồng là gì?"),
    "Notice Period To Terminate Renewal":("What is the notice period required to terminate renewal?", "Thời gian thông báo để chấm dứt gia hạn là bao lâu?"),
    "Governing Law":                     ("What is the governing law of this contract?",               "Luật điều chỉnh hợp đồng này là luật nào?"),
    "Most Favored Nation":               ("Does this contract contain a most favored nation clause?",  "Hợp đồng có điều khoản tối huệ quốc không?"),
    "Non-Compete":                       ("Does this contract contain a non-compete clause?",          "Hợp đồng có điều khoản không cạnh tranh không?"),
    "Exclusivity":                       ("Is there an exclusivity clause in this contract?",          "Hợp đồng có điều khoản độc quyền không?"),
    "No-Solicit Of Customers":           ("Does this contract restrict solicitation of customers?",    "Hợp đồng có cấm tiếp cận khách hàng không?"),
    "Competitive Restriction Exception": ("Are there exceptions to competitive restrictions?",         "Có ngoại lệ nào cho điều khoản hạn chế cạnh tranh không?"),
    "No-Solicit Of Employees":           ("Does this contract restrict solicitation of employees?",    "Hợp đồng có cấm tuyển dụng nhân viên của bên kia không?"),
    "Non-Disparagement":                 ("Is there a non-disparagement clause in this contract?",     "Hợp đồng có điều khoản không được bôi nhọ không?"),
    "Termination For Convenience":       ("Can this contract be terminated for convenience?",          "Hợp đồng có thể chấm dứt theo ý muốn không?"),
    "Rofr/Rofo/Rofn":                   ("Is there a right of first refusal or first offer clause?",  "Hợp đồng có điều khoản quyền ưu tiên mua không?"),
    "Change Of Control":                 ("What happens upon a change of control?",                    "Điều gì xảy ra khi có thay đổi quyền kiểm soát?"),
    "Anti-Assignment":                   ("Does this contract restrict assignment?",                   "Hợp đồng có điều khoản chống chuyển nhượng không?"),
    "Revenue/Profit Sharing":            ("Is there a revenue or profit sharing clause?",              "Hợp đồng có điều khoản chia sẻ doanh thu không?"),
    "Price Restrictions":                ("Are there any price restrictions in this contract?",        "Hợp đồng có điều khoản hạn chế giá không?"),
    "Minimum Commitment":                ("What is the minimum commitment in this contract?",          "Cam kết tối thiểu trong hợp đồng là gì?"),
    "Volume Restriction":                ("Are there volume restrictions in this contract?",           "Hợp đồng có hạn chế khối lượng không?"),
    "Ip Ownership Assignment":           ("Who owns the intellectual property in this contract?",      "Ai sở hữu tài sản trí tuệ trong hợp đồng này?"),
    "Joint Ip Ownership":                ("Is there joint intellectual property ownership?",           "Hợp đồng có điều khoản đồng sở hữu tài sản trí tuệ không?"),
    "License Grant":                     ("What license is granted in this contract?",                 "Hợp đồng cấp phép sử dụng gì?"),
    "Non-Transferable License":          ("Is the license non-transferable?",                          "Giấy phép có thể chuyển nhượng không?"),
    "Affiliate License-Licensor":        ("Can the licensor extend the license to affiliates?",        "Bên cấp phép có thể mở rộng giấy phép cho công ty liên kết không?"),
    "Affiliate License-Licensee":        ("Can the licensee extend the license to affiliates?",        "Bên được cấp phép có thể mở rộng cho công ty liên kết không?"),
    "Unlimited/All-You-Can-Eat-License": ("Is there an unlimited license granted?",                   "Hợp đồng có cấp phép sử dụng không giới hạn không?"),
    "Irrevocable Or Perpetual License":  ("Is the license irrevocable or perpetual?",                 "Giấy phép có vĩnh viễn hoặc không thể thu hồi không?"),
    "Source Code Escrow":                ("Is there a source code escrow clause?",                    "Hợp đồng có điều khoản ký quỹ mã nguồn không?"),
    "Post-Termination Services":         ("Are there post-termination service obligations?",           "Có nghĩa vụ dịch vụ nào sau khi chấm dứt hợp đồng không?"),
    "Audit Rights":                      ("Does this contract include audit rights?",                  "Hợp đồng có điều khoản quyền kiểm toán không?"),
    "Uncapped Liability":                ("Is there uncapped liability in this contract?",             "Hợp đồng có trách nhiệm pháp lý không giới hạn không?"),
    "Cap On Liability":                  ("What is the cap on liability in this contract?",            "Giới hạn trách nhiệm pháp lý trong hợp đồng là bao nhiêu?"),
    "Liquidated Damages":                ("Are there liquidated damages clauses?",                     "Hợp đồng có điều khoản bồi thường thiệt hại ấn định không?"),
    "Warranty Duration":                 ("What is the warranty duration in this contract?",           "Thời hạn bảo hành trong hợp đồng là bao lâu?"),
    "Insurance":                         ("What are the insurance requirements in this contract?",     "Yêu cầu bảo hiểm trong hợp đồng là gì?"),
    "Covenant Not To Sue":               ("Is there a covenant not to sue clause?",                   "Hợp đồng có điều khoản cam kết không khởi kiện không?"),
    "Third Party Beneficiary":           ("Are there third party beneficiaries in this contract?",     "Hợp đồng có bên thụ hưởng thứ ba không?"),
}

def extract_clause_type(raw_query):
    match = re.search(r'"([^"]+)"', raw_query)
    return match.group(1) if match else None

rows = []
for _, row in df.iterrows():
    clause = extract_clause_type(row["query"])
    if clause not in query_mapping:
        continue
    en_q, vi_q = query_mapping[clause]
    rows.append({"query": en_q, "passage": row["passage"], "label": row["label"]})
    rows.append({"query": vi_q, "passage": row["passage"], "label": row["label"]})

df_transformed = pd.DataFrame(rows)

positives = df_transformed[df_transformed["label"]==1][["query","passage"]].copy()
positives = positives.sample(frac=1, random_state=42).reset_index(drop=True)

split    = int(len(positives) * 0.8)
train_df = positives[:split]
val_df   = positives[split:]

print(f"Positives : {len(positives):,}")
print(f"Train     : {len(train_df):,}")
print(f"Val       : {len(val_df):,}")

Positives : 13,404
Train     : 10,723
Val       : 2,681


In [4]:
#Fine-tune
import torch, gc
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

gc.collect()
torch.cuda.empty_cache()
print(f"GPU: {torch.cuda.is_available()}")

train_examples = [
    InputExample(texts=[r["query"], r["passage"]])
    for _, r in train_df.iterrows()
]

model = SentenceTransformer("intfloat/multilingual-e5-base")

BATCH_SIZE = 16
EPOCHS     = 3

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)
train_loss       = losses.MultipleNegativesRankingLoss(model=model)
warmup_steps     = int(len(train_dataloader) * EPOCHS * 0.1)

print(f"Steps/epoch: {len(train_dataloader):,} | Total: {len(train_dataloader)*EPOCHS:,}")
print("Bắt đầu fine-tune...")

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    show_progress_bar=True,
    output_path="/kaggle/working/contract-e5-finetuned",
    checkpoint_path="/kaggle/working/checkpoints",
    checkpoint_save_steps=500,
)

print("✅ Fine-tune completed!")

/tmp/ipykernel_24/1437328796.py:3: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses


GPU: True


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Fine-tune completed!


In [5]:
from huggingface_hub import notebook_login
notebook_login()

In [6]:
model_ft = SentenceTransformer("/kaggle/working/contract-e5-finetuned")
model_ft.push_to_hub("d90nqm/contract-search-e5")
print("Model saved!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

HfHubHTTPError: Client error '401 Unauthorized' for url 'https://huggingface.co/api/repos/create' (Request ID: Root=1-6a43d420-6ea85d085d0ea9d02a7298d7;d3ae82b4-9d42-4255-a4cb-523acd648199)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

Invalid username or password.

In [ ]:
from huggingface_hub import HfApi
import os

api = HfApi()

# Tạo repo trước
api.create_repo(
    repo_id="d90nqm/contract-search-e5",
    private=True,
    exist_ok=True
)

# Upload từng file, bỏ qua README
model_path = "/kaggle/working/contract-e5-finetuned"

for filename in os.listdir(model_path):
    if filename == "README.md":
        continue
    
    filepath = os.path.join(model_path, filename)
    if os.path.isfile(filepath):
        print(f"Uploading {filename}...")
        api.upload_file(
            path_or_fileobj=filepath,
            path_in_repo=filename,
            repo_id="d90nqm/contract-search-e5",
        )

print("Uploaded!")

In [ ]:
from huggingface_hub import HfApi
import os

api = HfApi()

model_path = "/kaggle/working/contract-e5-finetuned"

for root, dirs, files in os.walk(model_path):
    for filename in files:
        if filename == "README.md":
            continue
        
        local_path = os.path.join(root, filename)
        # Tính đường dẫn tương đối trong repo
        repo_path  = os.path.relpath(local_path, model_path)
        
        print(f"Uploading {repo_path}...")
        api.upload_file(
            path_or_fileobj=local_path,
            path_in_repo=repo_path,
            repo_id="d90nqm/contract-search-e5",
        )

print("Uploaded!")

In [ ]:
from sentence_transformers import SentenceTransformer

model_check = SentenceTransformer("d90nqm/contract-search-e5")
test = model_check.encode(["payment terms"], normalize_embeddings=True)
print(f"✅ Load thành công! Vector shape: {test.shape}")

In [ ]:
#Chunk + index 1 hợp đồng
import chromadb
from sentence_transformers import SentenceTransformer

model_ft = SentenceTransformer("d90nqm/contract-search-e5")

def chunk_text(text, chunk_size=200, overlap=50):
    words  = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap
    return chunks

# Lấy 1 hợp đồng từ CUAD
sample_contract = cuad_raw["data"][0]
contract_name   = sample_contract["title"]
full_text       = " ".join([p["context"] for p in sample_contract["paragraphs"]])
chunks          = chunk_text(full_text, chunk_size=200, overlap=50)

print(f"Hợp đồng : {contract_name}")
print(f"Độ dài   : {len(full_text.split()):,} từ")
print(f"Số chunks: {len(chunks)}")

In [ ]:
#Embed và index
client = chromadb.Client()
try:
    client.delete_collection("contract_chunks")
except:
    pass

collection = client.create_collection("contract_chunks")

print(f"Đang embed {len(chunks)} chunks...")
embeddings = model_ft.encode(chunks, normalize_embeddings=True, show_progress_bar=True).tolist()

collection.add(
    embeddings=embeddings,
    documents=chunks,
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    metadatas=[{"chunk_index": i} for i in range(len(chunks))]
)

print(f"✅ Indexed {len(chunks)} chunks!")

In [ ]:
#Demo Gradio
import gradio as gr

def search_in_contract(query: str, top_k: int = 3):
    if not query.strip():
        return "⚠️ Vui lòng nhập câu hỏi"

    q_emb   = model_ft.encode([query], normalize_embeddings=True).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=top_k)

    output = f"### Kết quả cho: *\"{query}\"*\n\n"
    output += f"📄 `{contract_name}`\n\n---\n"

    for i, (doc, dist) in enumerate(zip(
        results["documents"][0],
        results["distances"][0]
    )):
        score = round(1 - dist, 3)
        bar   = "█" * int(score * 20)
        output += f"**Đoạn #{results['metadatas'][0][i]['chunk_index']+1}** · score: `{score}` {bar}\n\n"
        output += f"> {doc[:400]}\n\n---\n"

    return output

demo = gr.Interface(
    fn=search_in_contract,
    inputs=[
        gr.Textbox(
            label="Câu hỏi (tiếng Việt hoặc tiếng Anh)",
            placeholder="VD: payment terms / điều khoản chấm dứt / governing law",
            lines=2
        ),
        gr.Slider(1, 5, value=3, step=1, label="Số đoạn trả về")
    ],
    outputs=gr.Markdown(label="Kết quả"),
    title="📄 Contract Content Search",
    description=f"Tìm kiếm nội dung trong hợp đồng — hỗ trợ Việt / Anh",
    examples=[
        ["payment terms", 3],
        ["điều khoản chấm dứt hợp đồng", 3],
        ["governing law", 3],
        ["who are the parties", 3],
        ["thời hạn bảo hành", 3],
        ["liability cap", 3],
    ]
)

demo.launch(share=True)

In [ ]:
#Chunk nhỏ hơn + đúng prefix e5
client = chromadb.Client()
try:
    client.delete_collection("contract_chunks")
except:
    pass
collection = client.create_collection("contract_chunks")

def chunk_text_v2(text, chunk_size=100, overlap=20):
    """Chunk nhỏ hơn để mỗi đoạn tập trung 1 nội dung"""
    words  = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap
    return chunks

full_text = " ".join([p["context"] for p in sample_contract["paragraphs"]])
chunks_v2 = chunk_text_v2(full_text, chunk_size=100, overlap=20)

# Thêm prefix "passage: " — đây là yêu cầu của multilingual-e5
passages_to_index = [f"passage: {c}" for c in chunks_v2]

print(f"Chunks mới : {len(chunks_v2)} (thay vì {len(chunks)})")
print(f"Chunk mẫu  : {chunks_v2[38][:200]}")

embeddings = model_ft.encode(
    passages_to_index,
    normalize_embeddings=True,
    show_progress_bar=True
).tolist()

collection.add(
    embeddings=embeddings,
    documents=chunks_v2,  # lưu text gốc không có prefix
    ids=[f"chunk_{i}" for i in range(len(chunks_v2))],
    metadatas=[{"chunk_index": i} for i in range(len(chunks_v2))]
)

print(f"✅ Indexed {len(chunks_v2)} chunks!")

In [ ]:
#Cập nhật search function dùng đúng prefix e5
def search_in_contract_v2(query: str, top_k: int = 3):
    if not query.strip():
        return "⚠️ Vui lòng nhập câu hỏi"

    # Thêm prefix "query: " — đây là yêu cầu của multilingual-e5
    q_with_prefix = f"query: {query}"
    q_emb = model_ft.encode([q_with_prefix], normalize_embeddings=True).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=top_k)

    output = f"### Kết quả cho: *\"{query}\"*\n\n"
    output += f"📄 `{contract_name}`\n\n---\n"

    for i, (doc, dist) in enumerate(zip(
        results["documents"][0],
        results["distances"][0]
    )):
        score = round(1 - dist, 3)
        bar   = "█" * int(score * 20)
        output += f"**Đoạn #{results['metadatas'][0][i]['chunk_index']+1}** · score: `{score}` {bar}\n\n"
        output += f"> {doc[:400]}\n\n---\n"

    return output

demo2 = gr.Interface(
    fn=search_in_contract_v2,
    inputs=[
        gr.Textbox(
            label="Câu hỏi (tiếng Việt hoặc tiếng Anh)",
            placeholder="VD: payment terms / điều khoản chấm dứt / governing law",
            lines=2
        ),
        gr.Slider(1, 5, value=3, step=1, label="Số đoạn trả về")
    ],
    outputs=gr.Markdown(label="Kết quả"),
    title="📄 Contract Content Search v2",
    description=f"Tìm kiếm nội dung trong hợp đồng — hỗ trợ Việt / Anh",
    examples=[
        ["governing law", 3],
        ["payment terms", 3],
        ["điều khoản chấm dứt hợp đồng", 3],
        ["termination notice period", 3],
        ["thời hạn hợp đồng", 3],
        ["confidentiality", 3],
        ["liability", 3],
    ]
)

demo2.launch(share=True)

In [ ]:
for i, chunk in enumerate(chunks_v2):
    if "Governing Law" in chunk or "governing law" in chunk.lower():
        print(f"Chunk #{i+1}:")
        print(chunk)
        print("---")

In [ ]:
#Chunk theo section thay vì số từ
import re

client = chromadb.Client()
try:
    client.delete_collection("contract_chunks")
except:
    pass
collection = client.create_collection("contract_chunks")

def chunk_by_section(text):
    """
    Tách hợp đồng theo section number pattern
    VD: "6.1 ...", "6.2 ...", "Article 1", "SECTION 2"
    """
    # Pattern nhận diện đầu section
    section_pattern = r'(?=(?:\d+\.\d+[\s.]+[A-Z]|ARTICLE\s+\d+|SECTION\s+\d+|^\d+\.\s+[A-Z]))'
    
    parts = re.split(section_pattern, text, flags=re.MULTILINE)
    
    # Lọc bỏ đoạn quá ngắn (<20 từ)
    chunks = []
    for part in parts:
        part = part.strip()
        if len(part.split()) >= 20:
            chunks.append(part)
    
    return chunks

full_text  = " ".join([p["context"] for p in sample_contract["paragraphs"]])
chunks_sec = chunk_by_section(full_text)

print(f"Số sections: {len(chunks_sec)}")
print(f"\nVí dụ các section:")
for i, c in enumerate(chunks_sec[:5]):
    print(f"\n--- Section {i+1} ---")
    print(c[:150])

In [ ]:
#Index theo section + demo
passages_to_index = [f"passage: {c}" for c in chunks_sec]

embeddings = model_ft.encode(
    passages_to_index,
    normalize_embeddings=True,
    show_progress_bar=True
).tolist()

collection.add(
    embeddings=embeddings,
    documents=chunks_sec,
    ids=[f"sec_{i}" for i in range(len(chunks_sec))],
    metadatas=[{"chunk_index": i} for i in range(len(chunks_sec))]
)

print(f"✅ Indexed {len(chunks_sec)} sections!")

# Verify: tìm governing law
q_emb   = model_ft.encode(["query: governing law"], normalize_embeddings=True).tolist()
results = collection.query(query_embeddings=q_emb, n_results=3)

print("\n=== Test: governing law ===")
for i, (doc, dist) in enumerate(zip(results["documents"][0], results["distances"][0])):
    print(f"\nTop {i+1} (score: {round(1-dist,3)}):")
    print(doc[:300])

In [ ]:
# Tìm trong full_text thô
idx = full_text.find("Governing Law")
if idx >= 0:
    print("Tìm thấy tại vị trí:", idx)
    print(full_text[idx-100:idx+300])
else:
    print("Không tìm thấy 'Governing Law' trong full_text")

In [ ]:
#Fix section chunking
import re

client = chromadb.Client()
try:
    client.delete_collection("contract_chunks")
except:
    pass
collection = client.create_collection("contract_chunks")

def chunk_by_section_v2(text):
    # Tách theo pattern x.x hoặc số nguyên đầu dòng
    section_pattern = r'(\d+\.\d+\s+[A-Z][^\n]*?\.)' 
    
    # Tìm tất cả section headings và vị trí của chúng
    matches = list(re.finditer(r'\d+\.\d+\s+\w', text))
    
    chunks = []
    for i, match in enumerate(matches):
        start = match.start()
        end   = matches[i+1].start() if i+1 < len(matches) else len(text)
        chunk = text[start:end].strip()
        
        # Chỉ lọc bỏ nếu dưới 5 từ
        if len(chunk.split()) >= 5:
            chunks.append(chunk)
    
    return chunks

full_text  = " ".join([p["context"] for p in sample_contract["paragraphs"]])
chunks_sec = chunk_by_section_v2(full_text)

print(f"Số sections: {len(chunks_sec)}")

# Verify Governing Law có trong chunks không
for i, c in enumerate(chunks_sec):
    if "Governing Law" in c:
        print(f"\n✅ Tìm thấy tại Section #{i+1}:")
        print(c)

In [ ]:
# Cell — Index lại và test
passages_to_index = [f"passage: {c}" for c in chunks_sec]

embeddings = model_ft.encode(
    passages_to_index,
    normalize_embeddings=True,
    show_progress_bar=True
).tolist()

collection.add(
    embeddings=embeddings,
    documents=chunks_sec,
    ids=[f"sec_{i}" for i in range(len(chunks_sec))],
    metadatas=[{"chunk_index": i} for i in range(len(chunks_sec))]
)

print(f"✅ Indexed {len(chunks_sec)} sections!")

# Test 3 queries
test_queries = ["governing law", "luật điều chỉnh hợp đồng", "điều khoản chấm dứt"]

for query in test_queries:
    q_emb   = model_ft.encode([f"query: {query}"], normalize_embeddings=True).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=1)
    doc     = results["documents"][0][0]
    score   = round(1 - results["distances"][0][0], 3)
    status  = "✅" if "Governing Law" in doc or "termination" in doc.lower() or "chấm dứt" in doc.lower() else "❓"
    print(f"\n{status} Query: '{query}' (score: {score})")
    print(f"   → {doc[:200]}")

In [ ]:
# Cell — Gradio v3 final
import gradio as gr

def search_final(query: str, top_k: int = 3):
    if not query.strip():
        return "⚠️ Vui lòng nhập câu hỏi"

    q_emb   = model_ft.encode([f"query: {query}"], normalize_embeddings=True).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=top_k)

    output = f"### Kết quả cho: *\"{query}\"*\n\n"
    output += f"📄 `{contract_name}`\n\n---\n"

    for i, (doc, dist) in enumerate(zip(
        results["documents"][0],
        results["distances"][0]
    )):
        score  = round(1 - dist, 3)
        bar    = "█" * int(score * 20)
        sec_no = results["metadatas"][0][i]["chunk_index"] + 1
        output += f"**Section #{sec_no}** · score: `{score}` {bar}\n\n"
        output += f"> {doc[:500]}\n\n---\n"

    return output

demo_final = gr.Interface(
    fn=search_final,
    inputs=[
        gr.Textbox(
            label="Câu hỏi (tiếng Việt hoặc tiếng Anh)",
            placeholder="VD: governing law / điều khoản chấm dứt / payment terms",
            lines=2
        ),
        gr.Slider(1, 5, value=3, step=1, label="Số đoạn trả về")
    ],
    outputs=gr.Markdown(label="Kết quả"),
    title="📄 Contract Content Search — Final",
    description="Tìm kiếm theo điều khoản trong hợp đồng · Hỗ trợ Việt / Anh",
    examples=[
        ["governing law", 3],
        ["luật điều chỉnh hợp đồng", 3],
        ["payment terms", 3],
        ["điều khoản chấm dứt hợp đồng", 3],
        ["confidentiality", 3],
        ["bảo mật thông tin", 3],
        ["liability cap", 3],
        ["warranty", 3],
    ]
)

demo_final.launch(share=True)

In [ ]:
!pip install deep-translator -q

In [ ]:
# Dịch query và passage sang tiếng Việt
from deep_translator import GoogleTranslator

translator = GoogleTranslator(source='en', target='vi')

vi_pairs = []
for _, row in train_df.sample(n=500).iterrows():  # dịch 500 cặp trước
    try:
        query_vi   = translator.translate(row["query"])
        passage_vi = translator.translate(row["passage"][:500])  # giới hạn độ dài
        
        # Thêm cặp VI query → EN passage (cross-lingual)
        vi_pairs.append({"query": query_vi,   "passage": row["passage"]})
        # Thêm cặp VI query → VI passage (monolingual VI)
        vi_pairs.append({"query": query_vi,   "passage": passage_vi})
        # Thêm cặp EN query → VI passage (cross-lingual ngược)
        vi_pairs.append({"query": row["query"], "passage": passage_vi})
    except:
        continue

print(f"Tạo được {len(vi_pairs)} cặp tiếng Việt")

In [ ]:
# Thay thế model
model = SentenceTransformer("sentence-transformers/LaBSE")

# Test thử ngay
q = model.encode(["query: luật điều chỉnh hợp đồng"], normalize_embeddings=True)
p = model.encode(["passage: This Agreement shall be governed by the laws of Illinois"], normalize_embeddings=True)
print(f"Score: {(q @ p.T)[0][0]:.3f}")

In [ ]:
# Cell — Cài và dịch 500 cặp
!pip install deep-translator -q

from deep_translator import GoogleTranslator
import time

translator = GoogleTranslator(source='en', target='vi')

# Lấy 500 positive pairs từ train_df
sample = train_df.sample(n=500, random_state=42).reset_index(drop=True)

vi_pairs = []
errors   = 0

for i, row in sample.iterrows():
    try:
        query_vi   = translator.translate(row["query"][:200])
        passage_vi = translator.translate(row["passage"][:400])

        # 3 loại cặp cross-lingual
        vi_pairs.append({"query": query_vi,    "passage": row["passage"]})  # VI→EN
        vi_pairs.append({"query": query_vi,    "passage": passage_vi})      # VI→VI
        vi_pairs.append({"query": row["query"],"passage": passage_vi})      # EN→VI

        if i % 50 == 0:
            print(f"  {i}/500 xong...")
            time.sleep(1)  # tránh rate limit

    except Exception as e:
        errors += 1
        continue

vi_df = pd.DataFrame(vi_pairs)
print(f"\n✅ Tạo được {len(vi_df)} cặp tiếng Việt ({errors} lỗi)")
print(f"\nVí dụ VI→EN:")
print(f"  Query VI : {vi_df.iloc[0]['query']}")
print(f"  Passage  : {vi_df.iloc[0]['passage'][:100]}...")

In [ ]:
# Cell — Gộp data EN + VI rồi train lại
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
import torch, gc

# Gộp data gốc EN + data VI mới dịch
combined = pd.concat([
    train_df[["query","passage"]],  # ~10,723 cặp EN gốc
    vi_df[["query","passage"]]      # 1,500 cặp VI mới
], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"EN gốc  : {len(train_df):,}")
print(f"VI mới  : {len(vi_df):,}")
print(f"Tổng    : {len(combined):,}")

# Tạo InputExamples với prefix đúng chuẩn e5
train_examples_v2 = [
    InputExample(texts=[f"query: {r['query']}", f"passage: {r['passage']}"])
    for _, r in combined.iterrows()
]

gc.collect()
torch.cuda.empty_cache()
print(f"\nGPU: {torch.cuda.is_available()}")

model_v2 = SentenceTransformer("intfloat/multilingual-e5-base")

BATCH_SIZE = 16
EPOCHS     = 3

train_dataloader = DataLoader(train_examples_v2, shuffle=True, batch_size=BATCH_SIZE)
train_loss       = losses.MultipleNegativesRankingLoss(model=model_v2)
warmup_steps     = int(len(train_dataloader) * EPOCHS * 0.1)

print(f"Tổng steps : {len(train_dataloader) * EPOCHS:,}")
print(f"Warmup     : {warmup_steps}")
print("\nBắt đầu train lại...")

model_v2.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    show_progress_bar=True,
    output_path="/kaggle/working/contract-e5-v2",
    checkpoint_path="/kaggle/working/checkpoints-v2",
    checkpoint_save_steps=500,
)

print("✅ Train xong!")

In [ ]:
# Cell DUY NHẤT — chạy từ đầu sau restart
import json, re, random, time, gc
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
from deep_translator import GoogleTranslator

# ── 1. Load CUAD ──────────────────────────────────────────────
cuad_path = "/kaggle/input/datasets/ashyou09/contract-understanding-atticus-dataset-cuad/CUAD_v1.json"
with open(cuad_path) as f:
    cuad_raw = json.load(f)

pairs = []
for contract in cuad_raw["data"]:
    for paragraph in contract["paragraphs"]:
        context = paragraph["context"]
        for qa in paragraph["qas"]:
            pairs.append({
                "query":   qa["question"],
                "passage": context,
                "label":   1 if qa["answers"] else 0
            })
df = pd.DataFrame(pairs)
print(f"✅ Loaded: {len(df):,} pairs")

# ── 2. Transform queries ───────────────────────────────────────
query_mapping = {
    "Document Name":                     ("What is the name of this contract document?",              "Tên tài liệu hợp đồng này là gì?"),
    "Parties":                           ("Who are the parties involved in this contract?",            "Các bên tham gia hợp đồng này là ai?"),
    "Agreement Date":                    ("What is the agreement date of this contract?",              "Ngày ký kết hợp đồng là khi nào?"),
    "Effective Date":                    ("What is the effective date of this contract?",              "Ngày có hiệu lực của hợp đồng là khi nào?"),
    "Expiration Date":                   ("When does this contract expire?",                           "Hợp đồng này hết hạn vào ngày nào?"),
    "Renewal Term":                      ("What is the renewal term of this contract?",                "Điều khoản gia hạn hợp đồng là gì?"),
    "Notice Period To Terminate Renewal":("What is the notice period required to terminate renewal?", "Thời gian thông báo để chấm dứt gia hạn là bao lâu?"),
    "Governing Law":                     ("What is the governing law of this contract?",               "Luật điều chỉnh hợp đồng này là luật nào?"),
    "Most Favored Nation":               ("Does this contract contain a most favored nation clause?",  "Hợp đồng có điều khoản tối huệ quốc không?"),
    "Non-Compete":                       ("Does this contract contain a non-compete clause?",          "Hợp đồng có điều khoản không cạnh tranh không?"),
    "Exclusivity":                       ("Is there an exclusivity clause in this contract?",          "Hợp đồng có điều khoản độc quyền không?"),
    "No-Solicit Of Customers":           ("Does this contract restrict solicitation of customers?",    "Hợp đồng có cấm tiếp cận khách hàng không?"),
    "Competitive Restriction Exception": ("Are there exceptions to competitive restrictions?",         "Có ngoại lệ nào cho điều khoản hạn chế cạnh tranh không?"),
    "No-Solicit Of Employees":           ("Does this contract restrict solicitation of employees?",    "Hợp đồng có cấm tuyển dụng nhân viên của bên kia không?"),
    "Non-Disparagement":                 ("Is there a non-disparagement clause in this contract?",     "Hợp đồng có điều khoản không được bôi nhọ không?"),
    "Termination For Convenience":       ("Can this contract be terminated for convenience?",          "Hợp đồng có thể chấm dứt theo ý muốn không?"),
    "Rofr/Rofo/Rofn":                   ("Is there a right of first refusal or first offer clause?",  "Hợp đồng có điều khoản quyền ưu tiên mua không?"),
    "Change Of Control":                 ("What happens upon a change of control?",                    "Điều gì xảy ra khi có thay đổi quyền kiểm soát?"),
    "Anti-Assignment":                   ("Does this contract restrict assignment?",                   "Hợp đồng có điều khoản chống chuyển nhượng không?"),
    "Revenue/Profit Sharing":            ("Is there a revenue or profit sharing clause?",              "Hợp đồng có điều khoản chia sẻ doanh thu không?"),
    "Price Restrictions":                ("Are there any price restrictions in this contract?",        "Hợp đồng có điều khoản hạn chế giá không?"),
    "Minimum Commitment":                ("What is the minimum commitment in this contract?",          "Cam kết tối thiểu trong hợp đồng là gì?"),
    "Volume Restriction":                ("Are there volume restrictions in this contract?",           "Hợp đồng có hạn chế khối lượng không?"),
    "Ip Ownership Assignment":           ("Who owns the intellectual property in this contract?",      "Ai sở hữu tài sản trí tuệ trong hợp đồng này?"),
    "Joint Ip Ownership":                ("Is there joint intellectual property ownership?",           "Hợp đồng có điều khoản đồng sở hữu tài sản trí tuệ không?"),
    "License Grant":                     ("What license is granted in this contract?",                 "Hợp đồng cấp phép sử dụng gì?"),
    "Non-Transferable License":          ("Is the license non-transferable?",                          "Giấy phép có thể chuyển nhượng không?"),
    "Affiliate License-Licensor":        ("Can the licensor extend the license to affiliates?",        "Bên cấp phép có thể mở rộng giấy phép cho công ty liên kết không?"),
    "Affiliate License-Licensee":        ("Can the licensee extend the license to affiliates?",        "Bên được cấp phép có thể mở rộng cho công ty liên kết không?"),
    "Unlimited/All-You-Can-Eat-License": ("Is there an unlimited license granted?",                   "Hợp đồng có cấp phép sử dụng không giới hạn không?"),
    "Irrevocable Or Perpetual License":  ("Is the license irrevocable or perpetual?",                 "Giấy phép có vĩnh viễn hoặc không thể thu hồi không?"),
    "Source Code Escrow":                ("Is there a source code escrow clause?",                    "Hợp đồng có điều khoản ký quỹ mã nguồn không?"),
    "Post-Termination Services":         ("Are there post-termination service obligations?",           "Có nghĩa vụ dịch vụ nào sau khi chấm dứt hợp đồng không?"),
    "Audit Rights":                      ("Does this contract include audit rights?",                  "Hợp đồng có điều khoản quyền kiểm toán không?"),
    "Uncapped Liability":                ("Is there uncapped liability in this contract?",             "Hợp đồng có trách nhiệm pháp lý không giới hạn không?"),
    "Cap On Liability":                  ("What is the cap on liability in this contract?",            "Giới hạn trách nhiệm pháp lý trong hợp đồng là bao nhiêu?"),
    "Liquidated Damages":                ("Are there liquidated damages clauses?",                     "Hợp đồng có điều khoản bồi thường thiệt hại ấn định không?"),
    "Warranty Duration":                 ("What is the warranty duration in this contract?",           "Thời hạn bảo hành trong hợp đồng là bao lâu?"),
    "Insurance":                         ("What are the insurance requirements in this contract?",     "Yêu cầu bảo hiểm trong hợp đồng là gì?"),
    "Covenant Not To Sue":               ("Is there a covenant not to sue clause?",                   "Hợp đồng có điều khoản cam kết không khởi kiện không?"),
    "Third Party Beneficiary":           ("Are there third party beneficiaries in this contract?",     "Hợp đồng có bên thụ hưởng thứ ba không?"),
}

def extract_clause_type(q):
    m = re.search(r'"([^"]+)"', q)
    return m.group(1) if m else None

rows = []
for _, row in df.iterrows():
    clause = extract_clause_type(row["query"])
    if clause not in query_mapping:
        continue
    en_q, vi_q = query_mapping[clause]
    rows.append({"query": en_q, "passage": row["passage"], "label": row["label"]})
    rows.append({"query": vi_q, "passage": row["passage"], "label": row["label"]})

df_transformed = pd.DataFrame(rows)
positives = df_transformed[df_transformed["label"]==1][["query","passage"]].copy()
positives = positives.sample(frac=1, random_state=42).reset_index(drop=True)

split    = int(len(positives) * 0.8)
train_df = positives[:split]
val_df   = positives[split:]
print(f"✅ Train: {len(train_df):,} | Val: {len(val_df):,}")

# ── 3. Dịch 500 cặp sang tiếng Việt ──────────────────────────
translator = GoogleTranslator(source='en', target='vi')
sample_vi  = train_df.sample(n=500, random_state=42).reset_index(drop=True)
vi_pairs   = []

for i, row in sample_vi.iterrows():
    try:
        query_vi   = translator.translate(row["query"][:200])
        passage_vi = translator.translate(row["passage"][:400])
        vi_pairs.append({"query": query_vi,     "passage": row["passage"]})
        vi_pairs.append({"query": query_vi,     "passage": passage_vi})
        vi_pairs.append({"query": row["query"], "passage": passage_vi})
        if i % 100 == 0:
            print(f"  Dịch {i}/500...")
            time.sleep(1)
    except:
        continue

vi_df    = pd.DataFrame(vi_pairs)
combined = pd.concat([
    train_df[["query","passage"]],
    vi_df[["query","passage"]]
], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"✅ Tổng combined: {len(combined):,}")

# ── 4. Fine-tune ───────────────────────────────────────────────
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
gc.collect()
torch.cuda.empty_cache()
print(f"GPU: {torch.cuda.is_available()}")

train_examples = [
    InputExample(texts=[f"query: {r['query']}", f"passage: {r['passage']}"])
    for _, r in combined.iterrows()
]

model = SentenceTransformer("intfloat/multilingual-e5-base")

BATCH_SIZE = 8  # nhỏ hơn để tránh OOM
EPOCHS     = 3

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)
train_loss       = losses.MultipleNegativesRankingLoss(model=model)
warmup_steps     = int(len(train_dataloader) * EPOCHS * 0.1)

print(f"Steps/epoch : {len(train_dataloader):,}")
print(f"Total steps : {len(train_dataloader)*EPOCHS:,}")
print("Bắt đầu train...")

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    show_progress_bar=True,
    output_path="/kaggle/working/contract-e5-v2",
    checkpoint_path="/kaggle/working/checkpoints-v2",
    checkpoint_save_steps=500,
)

print("✅ Train xong!")


In [ ]:
# Cell — Chỉ chạy phần fine-tune (data đã có rồi)
import os
import gc
import torch
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
gc.collect()
torch.cuda.empty_cache()
print(f"GPU: {torch.cuda.is_available()}")

train_examples = [
    InputExample(texts=[f"query: {r['query']}", f"passage: {r['passage']}"])
    for _, r in combined.iterrows()
]

model = SentenceTransformer("intfloat/multilingual-e5-base")

BATCH_SIZE = 8
EPOCHS     = 3

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)
train_loss       = losses.MultipleNegativesRankingLoss(model=model)
warmup_steps     = int(len(train_dataloader) * EPOCHS * 0.1)

print(f"Steps/epoch : {len(train_dataloader):,}")
print(f"Total steps : {len(train_dataloader)*EPOCHS:,}")
print("Bắt đầu train...")

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    show_progress_bar=True,
    output_path="/kaggle/working/contract-e5-v2",
    checkpoint_path="/kaggle/working/checkpoints-v2",
    checkpoint_save_steps=500,
)

print("✅ Train xong!")

In [ ]:
#Lưu model đang có trong memory lên HuggingFace ngay
from huggingface_hub import HfApi
import os

# Lưu ra thư mục nhỏ hơn, không có checkpoint
model.save("/kaggle/working/contract-e5-v2-final")
print("✅ Saved locally")

# Upload lên HuggingFace
api = HfApi()
api.create_repo(repo_id="d90nqm/contract-search-e5-v2", private=True, exist_ok=True)

for root, dirs, files in os.walk("/kaggle/working/contract-e5-v2-final"):
    for filename in files:
        if filename == "README.md":
            continue
        local_path = os.path.join(root, filename)
        repo_path  = os.path.relpath(local_path, "/kaggle/working/contract-e5-v2-final")
        print(f"Uploading {repo_path}...")
        api.upload_file(
            path_or_fileobj=local_path,
            path_in_repo=repo_path,
            repo_id="d90nqm/contract-search-e5-v2",
        )

print("✅ Upload xong! d90nqm/contract-search-e5-v2")

In [ ]:
# Xóa checkpoints để giải phóng disk
import shutil, os

# Xóa các thư mục checkpoint tốn nhiều dung lượng nhất
for folder in ["/kaggle/working/checkpoints-v2", 
               "/kaggle/working/checkpoints",
               "/kaggle/working/contract-e5-finetuned"]:
    if os.path.exists(folder):
        shutil.rmtree(folder)
        print(f"✅ Đã xóa {folder}")

# Kiểm tra dung lượng còn lại
import shutil
total, used, free = shutil.disk_usage("/kaggle/working")
print(f"\nDisk free: {free/1024**3:.1f} GB")

In [ ]:
# Cell — Lưu và upload ngay sau khi có disk
model.save("/kaggle/working/contract-e5-v2-final")
print("✅ Saved locally")

from huggingface_hub import HfApi
api = HfApi()
api.create_repo(repo_id="d90nqm/contract-search-e5-v2", private=True, exist_ok=True)

for root, dirs, files in os.walk("/kaggle/working/contract-e5-v2-final"):
    for filename in files:
        if filename == "README.md":
            continue
        local_path = os.path.join(root, filename)
        repo_path  = os.path.relpath(local_path, "/kaggle/working/contract-e5-v2-final")
        print(f"Uploading {repo_path}...")
        api.upload_file(
            path_or_fileobj=local_path,
            path_in_repo=repo_path,
            repo_id="d90nqm/contract-search-e5-v2",
        )

print("✅ Upload xong!")

In [ ]:
from sentence_transformers import SentenceTransformer

model_v1 = SentenceTransformer("d90nqm/contract-search-e5")
model_v2 = SentenceTransformer("d90nqm/contract-search-e5-v2")

test_cases = [
    ("governing law",             "This Agreement shall be governed by the laws of Illinois."),
    ("luật điều chỉnh hợp đồng", "This Agreement shall be governed by the laws of Illinois."),
    ("điều khoản chấm dứt",      "Either party may terminate this Agreement upon 30 days notice."),
    ("payment terms",             "Payment shall be made within 30 days of invoice date."),
    ("bảo mật thông tin",        "Each party agrees to maintain confidentiality for 5 years."),
    ("thời hạn bảo hành",        "The warranty period is 12 months from the date of delivery."),
]

print(f"{'Query':<35} {'v1':>6} {'v2':>6} {'Tăng':>7}")
print("-" * 60)

for query, passage in test_cases:
    s1 = float(model_v1.encode([f"query: {query}"], normalize_embeddings=True) @
               model_v1.encode([f"passage: {passage}"], normalize_embeddings=True).T)
    s2 = float(model_v2.encode([f"query: {query}"], normalize_embeddings=True) @
               model_v2.encode([f"passage: {passage}"], normalize_embeddings=True).T)
    diff = s2 - s1
    flag = "✅" if diff > 0 else "❌"
    print(f"{query:<35} {s1:>6.3f} {s2:>6.3f}  {flag} {diff:>+.3f}")

In [ ]:
for query, passage in test_cases:
    q1 = model_v1.encode(f"query: {query}",   normalize_embeddings=True)
    p1 = model_v1.encode(f"passage: {passage}", normalize_embeddings=True)
    s1 = float(q1 @ p1)

    q2 = model_v2.encode(f"query: {query}",   normalize_embeddings=True)
    p2 = model_v2.encode(f"passage: {passage}", normalize_embeddings=True)
    s2 = float(q2 @ p2)

    diff = s2 - s1
    flag = "✅" if diff > 0 else "❌"
    print(f"{query:<35} {s1:>6.3f} {s2:>6.3f}  {flag} {diff:>+.3f}")

In [ ]:
import chromadb, gradio as gr
from sentence_transformers import SentenceTransformer

model_ft = SentenceTransformer("d90nqm/contract-search-e5-v2")

# Chunk theo section
import re
def chunk_by_section(text):
    matches = list(re.finditer(r'\d+\.\d+\s+\w', text))
    chunks  = []
    for i, match in enumerate(matches):
        start = match.start()
        end   = matches[i+1].start() if i+1 < len(matches) else len(text)
        chunk = text[start:end].strip()
        if len(chunk.split()) >= 5:
            chunks.append(chunk)
    return chunks

sample_contract = cuad_raw["data"][0]
contract_name   = sample_contract["title"]
full_text       = " ".join([p["context"] for p in sample_contract["paragraphs"]])
chunks_sec      = chunk_by_section(full_text)

# Index
client = chromadb.Client()
try:
    client.delete_collection("contract_chunks")
except:
    pass
collection = client.create_collection("contract_chunks")

embeddings = model_ft.encode(
    [f"passage: {c}" for c in chunks_sec],
    normalize_embeddings=True,
    show_progress_bar=True
).tolist()

collection.add(
    embeddings=embeddings,
    documents=chunks_sec,
    ids=[f"sec_{i}" for i in range(len(chunks_sec))],
    metadatas=[{"chunk_index": i} for i in range(len(chunks_sec))]
)
print(f"✅ Indexed {len(chunks_sec)} sections với model v2")

# Demo
def search_final(query: str, top_k: int = 3):
    if not query.strip():
        return "⚠️ Vui lòng nhập câu hỏi"
    q_emb   = model_ft.encode([f"query: {query}"], normalize_embeddings=True).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=top_k)
    output  = f"### Kết quả cho: *\"{query}\"*\n\n📄 `{contract_name}`\n\n---\n"
    for i, (doc, dist) in enumerate(zip(results["documents"][0], results["distances"][0])):
        score  = round(1 - dist, 3)
        bar    = "█" * int(score * 20)
        sec_no = results["metadatas"][0][i]["chunk_index"] + 1
        output += f"**Section #{sec_no}** · score: `{score}` {bar}\n\n> {doc[:500]}\n\n---\n"
    return output

gr.Interface(
    fn=search_final,
    inputs=[
        gr.Textbox(label="Câu hỏi (tiếng Việt hoặc tiếng Anh)",
                   placeholder="VD: governing law / điều khoản chấm dứt / payment terms",
                   lines=2),
        gr.Slider(1, 5, value=3, step=1, label="Số đoạn trả về")
    ],
    outputs=gr.Markdown(label="Kết quả"),
    title="📄 Contract Content Search v2",
    description="Model v2 — fine-tuned với data EN + VI · Score > 0.7",
    examples=[
        ["governing law", 3],
        ["luật điều chỉnh hợp đồng", 3],
        ["payment terms", 3],
        ["điều khoản chấm dứt hợp đồng", 3],
        ["bảo mật thông tin", 3],
        ["warranty duration", 3],
        ["thời hạn bảo hành", 3],
    ]
).launch(share=True)